In [82]:
import pandas as pd
import numpy as np
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import LabelEncoder
import torch.optim.lr_scheduler as lr_scheduler

In [83]:
torch.manual_seed(42)

#### **Loading data**

In [84]:
processed_dir = '../data/processed'
df = pd.read_parquet(f'{processed_dir}/master_data_v2.parquet')
item_counts = df.groupby('movieId').size()
warm_movies = item_counts[item_counts > 10].index
cf_df = df[df['movieId'].isin(warm_movies)].copy()
cf_df = cf_df.sort_values('datetime')

#### **Encoding IDs for PyTorch**

PyTorch Embeddings require inputs to be contiguous integers starting from 0

In [85]:
user_encoder = LabelEncoder()
item_encoder = LabelEncoder()

cf_df['user_idx'] = user_encoder.fit_transform(cf_df['userId'])
cf_df['item_idx'] = item_encoder.fit_transform(cf_df['movieId'])

num_users = cf_df['user_idx'].nunique()
num_items = cf_df['item_idx'].nunique()

#### **Time split**

In [86]:
split_index = int(len(cf_df) * 0.8)
train_df = cf_df.iloc[:split_index]
test_df = cf_df.iloc[split_index:]

#### **PyTorch Dataset and Dataloader**

In [87]:
class MovieLensDataset(Dataset):
    def __init__(self, users, items, ratings):
        self.users = torch.tensor(users, dtype=torch.long)
        self.items = torch.tensor(items, dtype=torch.long)
        self.ratings = torch.tensor(ratings, dtype=torch.float32)

    def __len__(self):
        return len(self.ratings)

    def __getitem__(self, idx):
        return self.users[idx], self.items[idx], self.ratings[idx]

In [88]:
train_dataset = MovieLensDataset(train_df['user_idx'].values, train_df['item_idx'].values, train_df['rating'].values)
test_dataset = MovieLensDataset(test_df['user_idx'].values, test_df['item_idx'].values, test_df['rating'].values)

train_loader = DataLoader(train_dataset, batch_size=4096, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=4096, shuffle=False)

#### **Building neural network**

In [89]:
class SimplifiedNeuMF(nn.Module):
    def __init__(self, num_users, num_items, embed_size=64):
        super().__init__() # Cleaner super() call in modern Python
        
        # 1. SHARED EMBEDDINGS (Halves the memory footprint!)
        self.user_embed = nn.Embedding(num_users, embed_size)
        self.item_embed = nn.Embedding(num_items, embed_size)
        
        # 2. SEQUENTIAL MLP BLOCK (Cleans up the init and forward pass)
        self.mlp_pipeline = nn.Sequential(
            nn.Linear(embed_size * 2, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU()
        )

        # Output: GMF features (embed_size) + MLP features (32)
        self.output = nn.Linear(embed_size + 32, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, user_indices, item_indices):
        # Grab the shared embeddings
        u = self.user_embed(user_indices)
        i = self.item_embed(item_indices)
        
        # --- The Math Path (GMF) ---
        # Python natively understands that multiplying two tensors means element-wise multiplication
        gmf_vector = u * i 
        
        # --- The Deep Learning Path (MLP) ---
        # Concatenate and pass through the entire Sequential block in one line
        mlp_vector = self.mlp_pipeline(torch.cat([u, i], dim=1))

        # --- Combine & Predict ---
        combined_vector = torch.cat([gmf_vector, mlp_vector], dim=1)
        prediction = self.output(combined_vector)
        
        # Scale to 0.5 - 5.0
        return (self.sigmoid(prediction) * 4.5 + 0.5).squeeze()

In [65]:
class TunedNeuMF(nn.Module):
    # TWEAK 1: Increased embed_size to 64
    def __init__(self, num_users, num_items, embed_size=64):
        super(TunedNeuMF, self).__init__()
        self.user_embed_gmf = nn.Embedding(num_users, embed_size)
        self.item_embed_gmf = nn.Embedding(num_items, embed_size)
        
        self.user_embed_mlp = nn.Embedding(num_users, embed_size)
        self.item_embed_mlp = nn.Embedding(num_items, embed_size)
        
        # TWEAK 1: Widened MLP Layers
        self.fc1 = nn.Linear(embed_size * 2, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 32)

        self.output = nn.Linear(embed_size + 32, 1)
        
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        self.sigmoid = nn.Sigmoid() # TWEAK 2: Added Sigmoid

    def forward(self, user_indices, item_indices):
        user_gmf = self.user_embed_gmf(user_indices)
        item_gmf = self.item_embed_gmf(item_indices)
        gmf_vector = torch.mul(user_gmf, item_gmf)
        
        user_mlp = self.user_embed_mlp(user_indices)
        item_mlp = self.item_embed_mlp(item_indices)
        mlp_vector = torch.cat([user_mlp, item_mlp], dim=1)
        
        mlp_vector = self.relu(self.fc1(mlp_vector))
        mlp_vector = self.dropout(mlp_vector)
        mlp_vector = self.relu(self.fc2(mlp_vector))
        mlp_vector = self.dropout(mlp_vector)
        mlp_vector = self.relu(self.fc3(mlp_vector))

        combined_vector = torch.cat([gmf_vector, mlp_vector], dim=1)
        prediction = self.output(combined_vector)
        
        # TWEAK 2: Scale output strictly to the 0.5 -> 5.0 range
        scaled_prediction = (self.sigmoid(prediction) * 4.5) + 0.5
        
        return scaled_prediction.squeeze()

In [90]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimplifiedNeuMF(num_users, num_items).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)

# TWEAK 3: Add Scheduler (Reduces LR by a factor of 0.5 if loss doesn't improve)
scheduler = lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=1)

#### **Training loop**

In [91]:
epochs = 8
print(f"Starting Tuned Training for {epochs} epochs...")

for epoch in range(epochs):
    model.train()
    total_loss = 0
    start_time = time.time()
    
    for users, items, ratings in train_loader:
        users, items, ratings = users.to(device), items.to(device), ratings.to(device)
        
        optimizer.zero_grad()
        predictions = model(users, items)
        
        loss = criterion(predictions, ratings)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
    avg_train_loss = total_loss / len(train_loader)
    
    # Step the scheduler at the end of the epoch
    scheduler.step(avg_train_loss)
    
    epoch_time = time.time() - start_time
    print(f"Epoch {epoch+1}/{epochs} | Train Loss (MSE): {avg_train_loss:.4f} | Time: {epoch_time:.2f}s")

Starting Tuned Training for 8 epochs...
Epoch 1/8 | Train Loss (MSE): 0.8765 | Time: 62.28s
Epoch 2/8 | Train Loss (MSE): 0.6772 | Time: 58.27s
Epoch 3/8 | Train Loss (MSE): 0.6573 | Time: 56.48s
Epoch 4/8 | Train Loss (MSE): 0.6440 | Time: 56.18s
Epoch 5/8 | Train Loss (MSE): 0.6324 | Time: 56.89s
Epoch 6/8 | Train Loss (MSE): 0.6258 | Time: 2257.78s
Epoch 7/8 | Train Loss (MSE): 0.6192 | Time: 56.70s
Epoch 8/8 | Train Loss (MSE): 0.6115 | Time: 57.92s


### **Evaluation**

In [92]:
model.eval()
test_predictions = []
test_actuals = []

with torch.no_grad():
    for users, items, ratings in test_loader:
        users, items = users.to(device), items.to(device)
        preds = model(users, items)
        # Clip predictions to 0.5 - 5.0 scale
        preds = torch.clamp(preds, min=0.5, max=5.0)
        test_predictions.extend(preds.cpu().numpy())
        test_actuals.extend(ratings.numpy())

In [93]:
rmse = np.sqrt(mean_squared_error(test_actuals, test_predictions))
print(f"Neural CF RMSE: {rmse:.4f}")
print(f"Improvement over SVD (0.9805): {0.9805 - rmse:.4f}")

Neural CF RMSE: 0.9611
Improvement over SVD (0.9805): 0.0194
